# YOLO 객체 탐지 — 사진과 영상 속 물체 찾기

**군-학 연계 프로그램 범용 교육 5차시** · 사전학습 모델로 사진·영상 속 물체를 찾고, 서명 탐지 모델을 직접 학습합니다.

## 사용 규칙 3가지
1. **위에서 아래로**, 셀 왼쪽의 ▶ 버튼을 하나씩 누릅니다. 셀이 끝나면 왼쪽 `[ ]` 안에 숫자가 생깁니다.
2. 바꿔 봐도 되는 줄은 `# <-- 바꿔 보세요` 표시가 있는 줄뿐입니다.
3. 빨간 글씨(에러)가 나오면 그대로 두고 손을 듭니다. 대부분 "셀 1을 안 돌림", "GPU를 안 켬", "순서를 건너뜀"입니다.

**오늘은 GPU가 필요합니다.** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**을 먼저 합니다. 셀 1이 GPU 이름을 알려 줍니다.

| 셀 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 1 | 준비(도구 설치 · 모델 · 샘플 사진 · 영상 · 데이터셋 내려받기) | 1~2분 |
| 2 | 사진 속 물체 찾기 | 몇 초 |
| 3 | 확신도 문턱값 바꾸기 · 특정 물체만 보기 | 바로 |
| 4 | 내 사진 올려서 찾기 | 바로 |
| 4b | 분할: 상자 대신 윤곽, 면적 재기 | 몇 초 |
| 5 | 영상 속 물체 찾기 | 약 30초 |
| 6 | 장면마다 사람 수 그래프 | 바로 |
| 7 | 작은 모델 vs 큰 모델 | 약 1분 |
| 8 | 서명 탐지 모델 학습 | 약 2분 |
| 9 | 학습 전 · 후 비교 (사진 · 영상) | 약 1분 |
| 10 | (선택) 학습 횟수를 줄이면 | 약 30초 |
| 11 | 내 모델 내려받기 | 바로 |

## 셀 1. 준비 — 도구를 설치하고 오늘 쓸 파일을 내려받습니다
▶를 누르고 "준비 완료"가 나올 때까지 기다립니다(1~2분). 중간에 나오는 회색 글씨는 무시합니다.

In [ ]:
# 셀 1. 준비 — 도구 설치 + 모델 · 샘플 사진 · 영상 · 데이터셋 내려받기 (1~2분)
import subprocess, sys, os, shutil, urllib.request, time, base64
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

import torch, cv2
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from PIL import Image as PILImage
from IPython.display import display, HTML

# (1) GPU 확인
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 없음 — 메뉴 [런타임 > 런타임 유형 변경 > T4 GPU > 저장] 후 이 셀을 다시 실행하세요. (CPU로도 되지만 느립니다)")

# (2) 사전학습 모델 2개 (작은 n, 중간 m)
YOLO("yolo26n.pt"); YOLO("yolo26m.pt")

# (3) 샘플 사진 2장 (Ultralytics 제공)
def fetch(url, path):
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (lecture notebook)"})
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            shutil.copyfileobj(r, f)
fetch("https://ultralytics.com/images/bus.jpg", "bus.jpg")
fetch("https://ultralytics.com/images/zidane.jpg", "zidane.jpg")

# (4) 샘플 영상: 도쿄 시부야 교차로 (Wikimedia Commons, Emran Kassim, CC BY 2.0) — 앞 15초만 잘라 둡니다
fetch("https://upload.wikimedia.org/wikipedia/commons/transcoded/f/f5/Shibuya_Crossing.ogv/Shibuya_Crossing.ogv.480p.vp9.webm",
      "shibuya.webm")
FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"], check=True)
    import imageio_ffmpeg; FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
def to_mp4(src, dst, seconds=None, width=None):
    """브라우저에서 재생되는 mp4(H.264)로 바꾼다."""
    cmd = [FFMPEG, "-y", "-loglevel", "error", "-i", src]
    if seconds: cmd += ["-t", str(seconds)]
    if width: cmd += ["-vf", f"scale={width}:-2"]
    cmd += ["-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "26", dst]
    subprocess.run(cmd, check=True)
if not os.path.exists("street.mp4"):
    to_mp4("shibuya.webm", "street.mp4", seconds=15)
cap = cv2.VideoCapture("street.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, 150); ok, frame = cap.read(); cap.release()
if ok: cv2.imwrite("crowd.jpg", frame)       # 영상의 한 장면(5초 지점)을 사진으로

# (5) 서명 탐지 데이터셋 (문서 사진 178장, Ultralytics 제공, 11MB)
DATA = check_det_dataset("signature.yaml")
VAL_DIR = str(DATA["val"])

# (6) 화면에 보여 주는 도우미
KO = {0:"사람",1:"자전거",2:"자동차",3:"오토바이",4:"비행기",5:"버스",6:"기차",7:"트럭",8:"배",9:"신호등",
      10:"소화전",11:"정지 표지판",12:"주차 미터기",13:"벤치",14:"새",15:"고양이",16:"개",17:"말",18:"양",19:"소",
      20:"코끼리",21:"곰",22:"얼룩말",23:"기린",24:"배낭",25:"우산",26:"핸드백",27:"넥타이",28:"여행가방",29:"프리스비",
      30:"스키",31:"스노보드",32:"공",33:"연",34:"야구 방망이",35:"야구 글러브",36:"스케이트보드",37:"서핑보드",38:"테니스 라켓",39:"병",
      40:"와인잔",41:"컵",42:"포크",43:"칼",44:"숟가락",45:"그릇",46:"바나나",47:"사과",48:"샌드위치",49:"오렌지",
      50:"브로콜리",51:"당근",52:"핫도그",53:"피자",54:"도넛",55:"케이크",56:"의자",57:"소파",58:"화분",59:"침대",
      60:"식탁",61:"변기",62:"TV",63:"노트북",64:"마우스",65:"리모컨",66:"키보드",67:"휴대폰",68:"전자레인지",69:"오븐",
      70:"토스터",71:"싱크대",72:"냉장고",73:"책",74:"시계",75:"꽃병",76:"가위",77:"곰인형",78:"헤어드라이어",79:"칫솔"}

def show(im_bgr, width=720):
    """탐지 결과 그림(BGR 배열)을 화면에 보여 준다."""
    im = PILImage.fromarray(im_bgr[..., ::-1])
    if im.width > width:
        im = im.resize((width, int(im.height * width / im.width)))
    display(im)

def report(r, names=None):
    """찾은 물체를 확신도 순서로 표로 찍는다."""
    names = names or (KO if len(r.names) == 80 else r.names)
    rows = sorted(zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist()), key=lambda x: -x[1])
    print(f"찾은 물체: {len(rows)}개")
    for c, p, (x1, y1, x2, y2) in rows:
        print(f"  {names[int(c)]:<8s} 확신도 {p*100:5.1f}%   상자 왼쪽위({x1:.0f},{y1:.0f}) 오른쪽아래({x2:.0f},{y2:.0f})")

def show_video(path, width=640):
    """mp4 영상을 노트북 안에서 재생한다."""
    data = base64.b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width="{width}" controls autoplay muted loop src="data:video/mp4;base64,{data}"></video>')

print("준비 완료")

## 셀 2. 사진 속 물체 찾기 — 사전학습 모델이 80종 물체를 찾습니다
모델 `yolo26n.pt`(YOLO26, 가장 작은 n 크기)는 이미 수십만 장으로 학습돼 있어 **사람 · 자동차 · 버스 · 개 · 컵** 같은 80종을 바로 찾습니다.

In [ ]:
# 셀 2. 사진 속 물체 찾기 — 샘플 사진 bus.jpg
model = YOLO("yolo26n.pt")
r = model.predict("bus.jpg", conf=0.25, verbose=False)[0]
report(r)
print(f"처리 시간: {r.speed['inference']:.0f} ms (한 장)")
show(r.plot())

## 셀 3. 확신도 문턱값 바꾸기 · 특정 물체만 보기
모델은 물체마다 **확신도**(0~100%)를 매깁니다. `CONF`보다 낮은 것은 버립니다. `ONLY`에 이름을 넣으면 그 물체만 남깁니다.

- 문턱값을 **0.05**로 낮추면 어떻게 되나요? **0.9**로 높이면?
- `ONLY = "사람"` 으로 바꾸면 사람만 셉니다.

In [ ]:
# 셀 3. 확신도 문턱값 · 특정 물체만 보기
CONF = 0.25          # <-- 바꿔 보세요: 0.05 ~ 0.9
ONLY = ""            # <-- 바꿔 보세요: "" (전부) 또는 "사람", "버스", "자동차" 같은 이름 하나
PHOTO = "bus.jpg"    # <-- 바꿔 보세요: "bus.jpg", "zidane.jpg", "crowd.jpg"

classes = [k for k, v in KO.items() if v == ONLY] if ONLY else None
if ONLY and not classes:
    print(f"'{ONLY}'은(는) 80종 목록에 없습니다. 목록:", ", ".join(KO.values()))
r = model.predict(PHOTO, conf=CONF, classes=classes, verbose=False)[0]
report(r)
show(r.plot())

## 셀 4. 내 사진 올려서 찾기
▶를 누르면 **파일 선택** 버튼이 나옵니다. PC에 저장한 사진(jpg · png)을 고릅니다. 인터넷에서 아무 사진이나 **이미지를 다른 이름으로 저장**해서 써도 됩니다.
개인정보가 있는 사진(신분증 · 가족 사진)은 올리지 않습니다. 파일을 고르지 않고 **취소**하면 샘플 사진 zidane.jpg로 진행합니다.

In [ ]:
# 셀 4. 내 사진 올려서 찾기
MY_CONF = 0.25       # <-- 바꿔 보세요
try:
    from google.colab import files
    up = files.upload()
    my_photo = next(iter(up)) if up else "zidane.jpg"
except ImportError:
    my_photo = "zidane.jpg"
print("사진:", my_photo)
r = model.predict(my_photo, conf=MY_CONF, verbose=False)[0]
report(r)
show(r.plot())

## 셀 4b. 분할(segmentation) — 상자 대신 윤곽을 따고, 면적을 잽니다
같은 사진에 **분할 모델**(`yolo26n-seg.pt`, 이름에 `-seg`만 붙었습니다)을 돌립니다. 왼쪽은 탐지(상자), 오른쪽은 분할(픽셀마다 색칠).
표에는 물체마다 **윤곽 면적 %**(색칠된 픽셀 ÷ 사진 전체 픽셀)와 **상자 면적 %**가 나옵니다. 상자는 물체 주변의 배경까지 포함하므로 윤곽보다 항상 큽니다.

In [ ]:
# 셀 4b. 분할 — 상자 대신 윤곽, 그리고 면적 (몇 초)
SEG_PHOTO = ""       # <-- 바꿔 보세요: "" 이면 셀 4에서 올린 내 사진, 아니면 "bus.jpg" · "zidane.jpg" · "crowd.jpg"
if not SEG_PHOTO:
    SEG_PHOTO = my_photo if "my_photo" in dir() else "bus.jpg"
import numpy as np
seg_model = YOLO("yolo26n-seg.pt")
rs = seg_model.predict(SEG_PHOTO, conf=0.25, retina_masks=True, verbose=False)[0]
H, W = rs.orig_shape
print(f"사진: {SEG_PHOTO} ({W}x{H} = {W*H:,} 픽셀)   찾은 물체: {len(rs.boxes)}개")
if rs.masks is None:
    print("윤곽을 딸 물체를 못 찾았습니다. SEG_PHOTO 를 다른 사진으로 바꿔 보세요.")
else:
    masks = rs.masks.data.cpu().numpy()              # (물체 수, H, W) 0/1 마스크
    rows = sorted(zip(rs.boxes.cls.tolist(), rs.boxes.conf.tolist(), masks, rs.boxes.xywh.tolist()), key=lambda x: -x[1])
    print(f"  {'물체':<8s} {'확신도':>6s}  {'윤곽 면적':>7s}  {'상자 면적':>7s}")
    for c, p, m, (_, _, bw, bh) in rows:
        print(f"  {KO[int(c)]:<8s} {p*100:5.1f}%   {m.mean()*100:6.1f}%   {bw*bh/(W*H)*100:6.1f}%")
    print(f"  색칠된 픽셀 합계: 사진의 {masks.max(axis=0).mean()*100:.1f}%  (나머지는 배경)")
left = model.predict(SEG_PHOTO, conf=0.25, verbose=False)[0].plot()
right = rs.plot(boxes=False, labels=False, conf=False)
show(np.hstack([left, right]), width=900)

## 셀 5. 영상 속 물체 찾기 — 영상은 사진의 연속입니다
15초짜리 교차로 영상(초당 30장, 450장)을 한 장씩 처리해 다시 영상으로 붙입니다(T4 기준 약 30초). 끝나면 결과 영상이 아래에 재생됩니다. 장면마다 **사람 수**도 함께 셉니다.

In [ ]:
# 셀 5. 영상 속 물체 찾기 — street.mp4 (15초) 처리 후 결과 영상 재생 (T4 약 30초)
VIDEO_CONF = 0.3     # <-- 바꿔 보세요

cap = cv2.VideoCapture("street.mp4")
fps = cap.get(cv2.CAP_PROP_FPS); W = int(cap.get(3)); H = int(cap.get(4)); cap.release()
out = cv2.VideoWriter("street_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
people = []                                   # 장면마다 사람 수
t0 = time.time()
for r in model.predict("street.mp4", conf=VIDEO_CONF, stream=True, verbose=False):
    people.append(int((r.boxes.cls == 0).sum()))
    out.write(r.plot())
out.release()
took = time.time() - t0
print(f"처리한 장면: {len(people)}장 / 걸린 시간 {took:.1f}초 / 초당 {len(people)/took:.0f}장 처리 (영상 자체는 초당 {fps:.0f}장)")
print(f"한 장면 속 사람 수: 최소 {min(people)}명, 최대 {max(people)}명, 평균 {sum(people)/len(people):.1f}명")
to_mp4("street_raw.mp4", "street_out.mp4")
show_video("street_out.mp4")

## 셀 6. 장면마다 사람 수 그래프
셀 5가 센 사람 수를 시간 순서로 그립니다. 사람이 몰리는 순간과 비는 순간이 보입니다. 매장 혼잡도 · 대기열 · 주차 공간 세기가 모두 이 방식입니다.

In [ ]:
# 셀 6. 장면마다 사람 수 그래프
import matplotlib.pyplot as plt
secs = [i / fps for i in range(len(people))]
plt.figure(figsize=(8, 3))
plt.plot(secs, people, lw=1.2)
plt.xlabel("seconds"); plt.ylabel("people in frame"); plt.title("People count per frame (street.mp4)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
busiest = max(range(len(people)), key=lambda i: people[i])
print(f"가장 붐빈 순간: {busiest/fps:.1f}초, {people[busiest]}명")

## 셀 7. 작은 모델 vs 큰 모델 — 속도와 정확도의 맞바꿈
같은 사진을 **n(작음)** 과 **m(중간)** 모델로 처리해 시간과 찾은 개수를 비교합니다. 큰 모델은 다이얼(가중치)이 많아 작은 물체를 더 잘 찾지만 느립니다.

In [ ]:
# 셀 7. 작은 모델 vs 큰 모델 (같은 사진, 같은 문턱값)
COMPARE_PHOTO = "crowd.jpg"   # <-- 바꿔 보세요: "crowd.jpg", "bus.jpg"
rows = []
for name in ["yolo26n.pt", "yolo26m.pt"]:
    m = YOLO(name)
    m.predict(COMPARE_PHOTO, verbose=False)                    # 준비 운동 (첫 실행은 느림)
    t0 = time.time()
    for _ in range(10):
        r = m.predict(COMPARE_PHOTO, conf=0.25, verbose=False)[0]
    ms = (time.time() - t0) / 10 * 1000
    params = sum(p.numel() for p in m.model.parameters()) / 1e6
    rows.append((name, params, ms, len(r.boxes)))
    print(f"[{name}] 다이얼 {params:.1f}백만 개 · 한 장 {ms:.1f} ms · 찾은 물체 {len(r.boxes)}개")
    show(r.plot(), width=640)
print()
print(f"{'모델':12s} {'다이얼(백만)':>10s} {'한 장(ms)':>10s} {'찾은 수':>8s}")
for name, params, ms, n in rows:
    print(f"{name:12s} {params:10.1f} {ms:10.1f} {n:8d}")

## 셀 8. 서명 탐지 모델 학습 — 80종에 없는 물체를 내 데이터로 가르칩니다
**서명(signature)** 은 80종 목록에 없습니다. 문서 사진 143장(+검증 35장)과 상자 좌표로 YOLO를 다시 학습합니다.
학습 한 바퀴(모든 사진을 한 번씩 보는 것)를 **에포크**라고 합니다. `EPOCHS = 20`이면 T4에서 약 2분 걸립니다. 진행 막대가 에포크마다 지나갑니다.

In [ ]:
# 셀 8. 서명 탐지 모델 학습 (T4 기준 약 2분)
EPOCHS = 20          # <-- 바꿔 보세요 (많을수록 오래 걸리고, 보통 더 정확해집니다)

RUNS = os.path.abspath("runs")            # 학습 결과가 저장되는 폴더
trainer = YOLO("yolo26n.pt")              # 80종을 아는 모델에서 출발 (처음부터 배우지 않는다)
t0 = time.time()
metrics = trainer.train(data="signature.yaml", epochs=EPOCHS, imgsz=640, batch=16,
                        project=RUNS, name="signature", exist_ok=True, plots=True)
BEST = os.path.join(RUNS, "signature", "weights", "best.pt")
print(f"\n학습 시간: {(time.time() - t0)/60:.1f}분")
print(f"검증 정확도(mAP50): {metrics.box.map50*100:.1f}%  — 검증 사진 35장에서 서명을 얼마나 정확히 찾았는가")
print("가장 좋은 모델 저장 위치:", BEST)
show(cv2.imread(os.path.join(RUNS, "signature", "results.png")), width=900)   # 왼쪽: 손실(낮을수록 좋음), 오른쪽: 정확도(높을수록 좋음)

## 셀 9. 학습 전 · 후 비교 — 같은 문서 사진, 다른 결과
학습 전 모델(`yolo26n.pt`)과 방금 학습한 모델(`best.pt`)에 **학습에 쓰지 않은** 검증 사진을 보여 줍니다. 이어서 서명이 담긴 짧은 영상도 처리합니다.

In [ ]:
# 셀 9. 학습 전 · 후 비교 (검증 사진 2장 + 영상)
import glob
before = YOLO("yolo26n.pt")
after = YOLO(BEST)
val_imgs = sorted(glob.glob(os.path.join(VAL_DIR, "*.jpg")))
TEST_IMGS = [val_imgs[0], val_imgs[16]]      # <-- 바꿔 보세요: 0~34 사이 번호

for p in TEST_IMGS:
    print("=" * 70); print(os.path.basename(p))
    rb = before.predict(p, conf=0.25, verbose=False)[0]
    ra = after.predict(p, conf=0.5, verbose=False)[0]
    print("[학습 전]", end=" "); report(rb)
    print("[학습 후]", end=" "); report(ra)
    both = cv2.hconcat([cv2.resize(rb.plot(), (640, 360)), cv2.resize(ra.plot(), (640, 360))])
    show(both, width=960)

# 서명 영상 (Ultralytics 제공, 11초)
fetch("https://ultralytics.com/assets/signature-s.mp4", "signature.mp4")
cap = cv2.VideoCapture("signature.mp4")
fps2 = cap.get(cv2.CAP_PROP_FPS); W2 = int(cap.get(3)); H2 = int(cap.get(4)); cap.release()
out = cv2.VideoWriter("signature_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps2, (W2, H2))
n = 0
for r in after.predict("signature.mp4", conf=0.5, stream=True, verbose=False):
    out.write(r.plot()); n += 1
out.release()
to_mp4("signature_raw.mp4", "signature_out.mp4", width=960)
print(f"영상 {n}장면 처리 완료")
show_video("signature_out.mp4", width=720)

## 셀 10. (선택) 학습 횟수를 줄이면 — 5 에포크짜리 모델과 비교
시간이 남으면 실행합니다. 같은 데이터로 **5 에포크만** 학습한 모델의 정확도를 셀 8의 모델과 나란히 봅니다.

In [ ]:
# 셀 10. (선택) 5 에포크 모델과 비교 (약 30초)
EPOCHS2 = 5          # <-- 바꿔 보세요
t0 = time.time()
m2 = YOLO("yolo26n.pt").train(data="signature.yaml", epochs=EPOCHS2, imgsz=640, batch=16,
                               project=RUNS, name="signature_short", exist_ok=True, plots=False)
print(f"\n{'모델':>12s} {'에포크':>6s} {'mAP50':>8s}")
print(f"{'셀 8':>12s} {EPOCHS:6d} {metrics.box.map50*100:7.1f}%")
print(f"{'셀 10':>12s} {EPOCHS2:6d} {m2.box.map50*100:7.1f}%   ({(time.time()-t0)/60:.1f}분)")

## 셀 11. 내 모델 내려받기
`best.pt` 한 파일이 오늘 만든 서명 탐지 모델입니다. 내려받아 두면 다른 PC의 노트북에서 `YOLO("best.pt")`로 바로 씁니다. 실습실 PC는 초기화되므로 Google Drive나 USB에 옮깁니다.

In [ ]:
# 셀 11. 내 모델 내려받기 (best.pt, 약 6MB)
try:
    from google.colab import files
    files.download(BEST)
except ImportError:
    print("Colab 밖에서는 이 파일을 직접 복사합니다:", BEST)